# SPECTRA Verification — Reviewer Tutorial

**Audience:** A working RF / communications engineer doing a critical review of someone else's verification work.  
**Goal:** Earn the reader's trust in the methodology behind `examples/verification/`.  
**Companion:** Every check in this notebook is also exposed as a top-level function in `tutorial_for_reviewers.py`; the script's `__main__` runs them all from the command line and exits non-zero on failure.

## §0 — How to read this tutorial

SPECTRA's verification suite groups every check into one of two tiers:

- **Property checks (`P*`)** — deterministic, fast, exact closed-form equalities or inequalities. Example: BPSK symbols lie on the real axis (max(|imag|) ≤ 1e-6). These run on every push and form the regression guard.
- **Performance checks (`S*`)** — statistical, slower, Monte-Carlo / sampling-bound. Example: BPSK BER vs Q(√(2·Eb/N0)) over [0, 6] dB, max |Δ| ≤ 0.8 dB. These run on demand or in nightly CI.

Every tolerance in the suite cites the literature (Proakis 2008, Levanon 2004, 3GPP TS 38.104, ITU-R SM.328, etc.). Citation keys like `proakis2008:§4.3.2` resolve to bibliography entries in `examples/verification/REFERENCES.md`. Any tolerance presented in this tutorial that doesn't have an inline derivation is one whose citation we trust without further comment.

**How to read the regression catalogue tables.** Each waveform section ends with a table of injected faults: a baseline measurement, then a series of rows where the signal has been deliberately corrupted. The point isn't just "the check fires" — it's also *which* checks fire on *which* faults. Some faults are caught by the PSD check but invisible to BER; some are the other way around. The layering is intentional and is the most important argument for why the suite isn't a single number.

## §1 — The suite has caught real bugs

Before walking through the methodology, two pieces of evidence that this isn't hypothetical.

### Bug 1 — GMSK `h_eff = 0.5/sps` → `h = 0.5`

**Symptom.** The pre-fix `GMSK.generate` in `python/spectra/waveforms/fsk.py` used zero-insertion upsampling and a sum-normalised Gaussian filter; the resulting effective modulation index was `h_eff = 0.5/sps = 0.0625` (for sps = 8) instead of the textbook `h = 0.5`. Frequency deviation was 8× smaller than standard MSK; the Laurent expansion didn't apply; the MSK BER curve didn't apply; the OBW was a sixth of the GSM reference.

**How the suite caught it.** `verify_gmsk.py::P2` measures the steady-state per-symbol phase change on a constant-bit stream and asserts it equals `π · h = π/2 rad` within 1 %. On the buggy code the measurement was `π · 0.0625 ≈ 0.196 rad` — a factor of 8 off. The check is the exact same shape we use for the BPSK constellation check below: closed-form, deterministic, citation-grounded.

**Fix.** PR #4 (commit `f034fb6`): switch to repeat-upsampling, matching `MSK.generate` and `FSK.generate`. One line.

### Bug 2 — 16-QAM row-major → Gray-coded labelling

**Symptom.** `build_qam_constellation` in `rust/src/modulators.rs` swept the I/Q grid in row-major order, so adjacent integer labels were not physical neighbours. A single-symbol error in a high-SNR AWGN channel could flip up to `log₂(M) = 4` bits at once (e.g., labels 3 and 4 differ in 3 bits, not 1). The BER↔SER relationship deviated from `BER ≈ SER/log₂(M)` by up to a factor of `log₂(M)` at moderate-to-high SNR.

**How the suite caught it.** The old `verify_qam16.py` had a deliberate `# P3 — Gray adjacency (SKIPPED: ...)` block: the verifier documented the defect rather than asserting it away. After the fix, P3 became a real check: every nearest-neighbour pair must have Gray-adjacent labels (popcount(label_a XOR label_b) == 1). And a new S3 check confirms `|BER − SER/log₂(M)| ≤ 5e-3` at Eb/N0 = 11 dB — after the fix the measurement is `~1.25e-6`, essentially exact.

**Fix.** PR #6 (commit `85a4154`): Gray-code each I/Q axis independently, place point at `constellation[(gray(i) << n) | gray(j)]`. Five lines.

**The takeaway.** These bugs were caught by the exact kinds of checks we're about to walk through for BPSK, OFDM, and Barker-13. The methodology generalises.

## Setup

Switch `FULL = True` to run publication-grade sample sizes (slow). Default is `FULL = False` (fast mode, ≤ 30 s).

In [ ]:
import sys
from pathlib import Path

# Make the example-local modules importable.
# The notebook lives in examples/verification/; the kernel cwd may be
# the worktree root (nbclient/pytest) or examples/verification/ (nbconvert).
_HERE = Path('.').resolve()
for cand in [
    _HERE,                                          # cwd IS examples/verification/
    _HERE / 'examples' / 'verification',            # cwd is worktree root
    _HERE.parent / 'examples' / 'verification',     # cwd is one level above root
    _HERE.parent.parent / 'examples' / 'verification',  # cwd is two levels above
]:
    if (cand / 'tutorial_for_reviewers.py').exists():
        sys.path.insert(0, str(cand))
        break

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

import tutorial_for_reviewers as tut
import _tutorial_regressions as reg

FULL = False  # toggle True for publication-grade Monte Carlos
print(f'Setup complete. FULL = {FULL}')